<a href="https://colab.research.google.com/github/gertdreyer/MIT808-standard-deviants/blob/main/MIT808_Data_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Dependencies

In [ ]:
!pip install boto3

!pip install pyiceberg[hive] pandas pyarrow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for thrift: filename=thrift-0.22.0-cp312-cp312-linux_x86_64.whl size=413530 sha256=4d8de1b83244945bf1aad84c05d1df5d8f3aea00fa3a77a0b64abc6fe81118d8
  Stored in directory: /root/.cache/pip/wheels/03/30/a2/6e53de46f9d8f098a6b7c4a57dbdcafa5f11e87274e139f50b
Successfully built thrift


In [ ]:


!pip install scrapling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.5/116.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 24.5 MB/s eta 0:00:00


In [ ]:
# set env
import os
from google.colab import userdata
os.environ['PYICEBERG_CATALOG__DEFAULT__S3__ACCESS_KEY_ID'] = userdata.get('PYICEBERG_CATALOG__DEFAULT__S3__ACCESS_KEY_ID')
os.environ['PYICEBERG_DOWNCAST_NS_TIMESTAMP_TO_US_ON_WRITE'] = 'true'
os.environ['PYICEBERG_CATALOG__DEFAULT__S3__SECRET_ACCESS_KEY'] = userdata.get('PYICEBERG_CATALOG__DEFAULT__S3__SECRET_ACCESS_KEY')

In [ ]:
from pyiceberg.catalog import load_catalog, rest
import pandas as pd
import pyarrow as pa
from datetime import datetime
from pyiceberg.expressions import EqualTo, In


In [ ]:
!mkdir -p warehouse

# Initialize Warehouse and S3 connection

In [ ]:
import boto3
s3_client = boto3.client(
    's3',
    endpoint_url="https://15c46705015d147b9bbf2d532ce26495.r2.cloudflarestorage.com" ,
    aws_access_key_id=userdata.get('PYICEBERG_CATALOG__DEFAULT__S3__ACCESS_KEY_ID'),
    aws_secret_access_key=userdata.get('PYICEBERG_CATALOG__DEFAULT__S3__SECRET_ACCESS_KEY'),
)


In [ ]:
warehouse_path = "/content/warehouse"
s3_uri = "https://15c46705015d147b9bbf2d532ce26495.r2.cloudflarestorage.com"

warehouse_path = "/content/warehouse"
s3_uri = "https://15c46705015d147b9bbf2d532ce26495.r2.cloudflarestorage.com"
catalog = rest.RestCatalog(
    name="default",
    warehouse="15c46705015d147b9bbf2d532ce26495_standard-deviants-storage",
    uri="https://catalog.cloudflarestorage.com/15c46705015d147b9bbf2d532ce26495/standard-deviants-storage",
    token=userdata.get('CLOUDFLARE_API_KEY'),
)

if not catalog.namespace_exists("default"):
  catalog.create_namespace("default")



# Create table using PyArrow schema
q_a_schema = pa.schema([
      ("created_at", pa.timestamp('s'),),
      ("updated_at", pa.timestamp('s'),),
      ("id",pa.int64(),),
      ("minister_id", pa.int64(),),
      ("code", pa.string(),),
      ("question_number", pa.int64(),),
      ("house_id", pa.int64(),),
      ("written_number", pa.int64(),),
      ("oral_number", pa.int64(),),
      ("president_number", pa.int64(),),
      ("deputy_president_number", pa.int64(),),
      ("answer_type", pa.string(),),
      ("date", pa.string(),),
      ("year", pa.int64(),),
      ("question", pa.string(),),
      ("answer", pa.string(),),
      ("question_to_name", pa.string(),),
      ("translated", pa.bool_(),),
      ("asked_by_name", pa.string(),),
      ("asked_by_member_id", pa.int64(),),
      ("source_file_id", pa.int64(),),
      ("url", pa.string(),),
      ("source_file.id", pa.int64(),),
      ("source_file.title", pa.string(),),
      ("source_file.file_mime", pa.string(),),
      ("source_file.file_path", pa.string(),),
      ("source_file.file_bytes", pa.int64(),),
      ("source_file.origname", pa.string(),),
      ("source_file.description", pa.string(),),
      ("source_file.playtime", pa.string(),),
      ("source_file.url", pa.string()),
      ("retrieved_at", pa.timestamp('s'),)
    ])
if not catalog.table_exists("default.questions_and_answers"):
  q_a_table = catalog.create_table("default.questions_and_answers", schema=q_a_schema)
else:
  q_a_table = catalog.load_table("default.questions_and_answers")

def retrieve_question_by_id(idents: list[str]):
  rows = q_a_table.scan(row_filter=In("id", idents)).to_pandas()
  return rows

# Questions and Answers

In [ ]:
import requests
from datetime import UTC
url = "https://api.pmg.org.za/committee-question/"
while True:
  print(url)
  if url is None:
    break
  resp = requests.get(url)
  respj = resp.json()
  retrieved_at = datetime.utcnow()
  results = respj["results"]
  page_data = []
  for jdata in results:
    if "source_file" in jdata:
      sf = jdata["source_file"]
    else:
      sf = {}
      jdata["source_file"] = {}
    jdata = {**jdata, **{f"source_file.{k}": v for k,v in sf.items()}, "retrieved_at" : retrieved_at}
    jdata["created_at"] = datetime.fromisoformat(jdata["created_at"])
    jdata["updated_at"] = datetime.fromisoformat(jdata["updated_at"])
    del jdata["source_file"]
    if "minister" in jdata:
      del jdata["minister"]
    del jdata["house"]
    page_data.append(jdata)
  check_db = retrieve_question_by_id([r["id"] for r in page_data])
  check_db_rows = {r['id']: r['updated_at'] for (_, r) in check_db.iterrows()}
  docs_to_delete = []
  docs_to_append = []
  for row in page_data:
    if row['id'] in check_db_rows and check_db_rows[row['id']].replace(tzinfo=UTC) < row['updated_at'].replace(microsecond=0):
      # delete stale record and reinsert
      docs_to_delete.append(row['id'])
      docs_to_append.append(row)
      print(f"Id {row["id"]} {row["date"]} has newer content ours {check_db_rows[row['id']]} < theirs {row['updated_at']}")
    elif row['id'] in check_db_rows:
      print(f"Id {row["id"]} Already seen ")
      pass
    else:
      docs_to_append.append(row)
      print(f"Id {row['id']} {row["date"]} unseen")
  url = respj["next"]
  if len(docs_to_delete) == 0 and len(docs_to_append) == 0:
    continue
  elif len(docs_to_append) == 0:
    continue
  elif len(docs_to_delete) > 0:
    q_a_table.delete(In("id", docs_to_delete))
  app_tab = pa.Table.from_pylist(docs_to_append, schema=q_a_schema)
  # app_tab.validate(full=True)
  q_a_table.append(app_tab)


In [ ]:
## Show loaded data

In [ ]:
rows = q_a_table.scan().to_pandas()
rows

,created_at,updated_at,id,minister_id,code,question_number,house_id,written_number,oral_number,president_number,...,source_file.id,source_file.title,source_file.file_mime,source_file.file_path,source_file.file_bytes,source_file.origname,source_file.description,source_file.playtime,source_file.url,retrieved_at
0,2025-01-17 11:44:29,2025-01-17 11:46:05,27814,4.0,NW1226,1226,3,1226.0,NaN,NaN,...,183055.0,None,application/vnd.openxmlformats-officedocument....,RNW1226-241224.docx,125205.0,None,None,None,https://pmg.org.za/files/RNW1226-241224.docx,2026-03-14 20:51:44
1,2025-01-17 11:51:59,2025-01-17 11:52:28,27819,27.0,NW1590,1590,3,1590.0,NaN,NaN,...,183061.0,None,application/vnd.openxmlformats-officedocument....,RNW1590-241224.docx,83005.0,None,None,None,https://pmg.org.za/files/RNW1590-241224.docx,2026-03-14 20:51:44
2,2025-01-17 11:47:13,2025-01-17 11:48:28,27815,4.0,NW1450,1450,3,1450.0,NaN,NaN,...,183056.0,None,application/vnd.openxmlformats-officedocument....,RNW1450-241224.docx,121747.0,None,None,None,https://pmg.org.za/files/RNW1450-241224.docx,2026-03-14 20:51:44
3,2025-01-17 10:56:18,2025-01-17 10:56:33,27803,4.0,NW1674,1674,3,1674.0,NaN,NaN,...,183041.0,None,application/vnd.openxmlformats-officedocument....,RNW1674-241224.docx,116418.0,None,None,None,https://pmg.org.za/files/RNW1674-241224.docx,2026-03-14 20:51:44
4,2025-01-17 11:49:55,2025-01-17 11:50:22,27817,4.0,NW1485,1485,3,1485.0,NaN,NaN,...,183059.0,None,application/vnd.openxmlformats-officedocument....,RNW1485-241224.docx,125859.0,None,None,None,https://pmg.org.za/files/RNW1485-241224.docx,2026-03-14 20:51:44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8295,2026-03-11 10:13:37,2026-03-11 10:40:59,36047,30.0,NW309,309,3,309.0,NaN,NaN,...,201518.0,None,application/vnd.openxmlformats-officedocument....,RNW309-2026-03-10.docx,17060.0,None,None,None,https://pmg.org.za/files/RNW309-2026-03-10.docx,2026-03-14 19:48:51
8296,2026-03-12 13:47:37,2026-03-12 13:48:30,36088,30.0,NW312,312,3,312.0,NaN,NaN,...,201616.0,None,application/vnd.openxmlformats-officedocument....,RNW312-260310.docx,106960.0,None,None,None,https://pmg.org.za/files/RNW312-260310.docx,2026-03-14 19:48:51
8297,2026-03-12 12:43:53,2026-03-12 12:44:25,36073,30.0,NW725,725,3,725.0,NaN,NaN,...,201600.0,None,application/vnd.openxmlformats-officedocument....,RNW725-260310.docx,2530560.0,None,None,None,https://pmg.org.za/files/RNW725-260310.docx,2026-03-14 19:48:51
8298,2026-03-12 13:17:33,2026-03-12 13:18:14,36076,20.0,NW330,330,3,330.0,NaN,NaN,...,201604.0,None,application/vnd.openxmlformats-officedocument....,RNW330-260310.docx,22934.0,None,None,None,https://pmg.org.za/files/RNW330-260310.docx,2026-03-14 19:48:51


# Proceedings
## Imports

In [ ]:
!pip install playwright
!playwright install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 56.7 MB/s eta 0:00:00
(node:3815) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 0.0s167.3 MiB [] 0% 118.0s167.3 MiB [] 0% 378.7s167.3 MiB [] 0% 432.8s167.3 MiB [] 0% 347.2s167.3 MiB [] 0% 299.7s167.3 MiB [] 0% 269.5s167.3 MiB [] 0% 249.4s167.3 MiB [] 0% 233.2s167.3 MiB [] 0% 218.9s167.3 MiB [] 0% 205.4s167.3 MiB [] 0% 193.9s167.3 MiB [] 0% 170.5s167.3 MiB [] 0% 153.6s167.3 MiB [] 0% 140.4s167.3 MiB [] 0% 130.4s167.3 MiB [] 0% 122.3s167.3 MiB [] 0% 112.7s167.3 MiB [] 0% 102.9s167.3 MiB [] 0% 93.5s167.3 MiB [] 0% 89.2s167.3 MiB [] 0% 86.1s167.3 MiB [] 0% 78.4s167.3 MiB [] 0% 70.5s167.3 MiB [] 0% 65.7s167.3 MiB [] 0% 63.7s167.3 MiB [] 1% 62.9s167.3 MiB [] 1% 59.4s167.3 MiB [] 1% 

In [ ]:
import asyncio
import requests
import fitz
import pytesseract
from PIL import Image
import io
from playwright.async_api import async_playwright
import pandas as pd


## Initialize Warehouse Connection

In [ ]:
warehouse_path = "/content/warehouse"
s3_uri = "https://15c46705015d147b9bbf2d532ce26495.r2.cloudflarestorage.com"
catalog = rest.RestCatalog(
    name="default",
    warehouse="15c46705015d147b9bbf2d532ce26495_standard-deviants-storage",
    uri="https://catalog.cloudflarestorage.com/15c46705015d147b9bbf2d532ce26495/standard-deviants-storage",
    token=userdata.get('CLOUDFLARE_API_KEY'),
)
if not catalog.namespace_exists("default"):
  catalog.create_namespace("default")

table_name = "default.proceedings"

# Drop table if it exists
if catalog.table_exists(table_name):
    catalog.drop_table(table_name)
    print(f"{table_name} dropped.")


# Create table using PyArrow schema
proceedings_schema = pa.schema([
    ("id", pa.int64()),
    ("uvimba_id", pa.string()),
    ("document_name", pa.string()),
    ("house", pa.string()),
    ("language", pa.string()),
    ("date", pa.string()),
    ("document_type", pa.string()),
    ("file_location", pa.string()),
    ("pdf_url", pa.string()),
    ("pdf_text", pa.string()),
])

if not catalog.table_exists(table_name):
  proceedings_table = catalog.create_table(table_name, schema=proceedings_schema)
else:
  proceedings_table = catalog.load_table(table_name)



default.proceedings dropped.


In [ ]:
import fitz  # PyMuPDF
import pytesseract
from pdf2image import convert_from_bytes

In [ ]:
def extract_pdf_text(url, stop_page=70):
    # Fetch PDF from URL
    import requests
    pdf_bytes = requests.get(url).content

    # Open PDF with fitz
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    text_output = ""

    for i, page in enumerate(doc):
        if i + 1 > stop_page:  # Stop at the specified page
            break

        page_text = page.get_text()

        # If the text contains garbage like '(cid:', apply OCR
        if page_text and '(cid:' in page_text:
            print(f"Page {i + 1} is an annexure, applying OCR...")

            # Convert the PDF page to image
            images = convert_from_bytes(pdf_bytes, first_page=i+1, last_page=i+1, dpi=300)
            for img in images:
                ocr_text = pytesseract.image_to_string(img)
                text_output += ocr_text + "\n\n"
        else:
            text_output += page_text + "\n\n"

    return text_output

In [ ]:
import requests
import re
from io import BytesIO

def sanitize_proceedings_filename(name: str) -> str:
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = name.replace(" ", "_")
    return name

from io import BytesIO
import requests

def upload_documents_to_s3(documents, s3_client, bucket_name):

    uploaded = 0

    for doc in documents:
        try:

            pdf_url = doc["pdf_url"]
            doc_id = doc["id"]

            filename = f"{doc_id}.pdf"
            s3_key = f"proceedings/{filename}"

            print(f"Uploading {filename}...")

            response = requests.get(pdf_url)

            if response.status_code != 200:
                print(f"Failed to download {pdf_url}")
                continue

            file_bytes = BytesIO(response.content)

            s3_client.upload_fileobj(
                file_bytes,
                bucket_name,
                s3_key
            )

            uploaded += 1

        except Exception as e:
            print(f"Upload failed for {doc['document_name']}: {e}")

    print(f"{uploaded} files uploaded to s3://{bucket_name}/proceedings/")

## Hansard

In [ ]:
import requests
import pandas as pd
import pyarrow as pa

url = "https://www.parliament.gov.za/docsjson/hansard"

params = {
    "queries[type]": "hansard",
    "sorts[date]": "-1",
    "page": 1,
    "perPage": 380,
    "offset": 0
}

print("Fetching Hansard records from API...")

response = requests.get(url, params=params)
data = response.json()

records = data["records"]

documents = []

for record in records:

    name = record["name"]
    house = record["house"]
    language = record["language"]
    date = record["date"]
    doc_type = record["type"]
    file_location = record["file_location"]

    pdf_url = f"https://www.parliament.gov.za/storage/app/media/Docs/{file_location}"

    print("Name:", name)
    print("House:", house)
    print("Language:", language)
    print("Date:", date)
    print("Type:", doc_type)
    print("File location:", file_location)
    print("PDF URL:", pdf_url)

    text = extract_pdf_text(pdf_url)

    documents.append({
        "id": record["id"],
        "uvimba_id": record["uVIMBA_id"],
        "document_name": name,
        "house": house,
        "language": language,
        "date": date,
        "document_type": doc_type,
        "file_location": file_location,
        "pdf_url": pdf_url,
        "pdf_text": text
    })


if documents:

    df = pd.DataFrame(documents)

    arrow_table = pa.Table.from_pandas(df)

    print("appending to proceedings table")

    proceedings_table.append(arrow_table)

    print(f"{len(df)} Hansard documents inserted into Iceberg table.")

    s3_client.upload_file(local_file_path, bucket_name, s3_object_key)

    print(f"Catalog uploaded to s3://{bucket_name}/{s3_object_key}")

    upload_documents_to_s3(documents, s3_client, bucket_name)

    rows = proceedings_table.scan().to_pandas()

    print("\nData currently in Hansard Iceberg table:\n")
    print(rows)

Fetching Hansard records from API...
Name: HAN-MPS-NA(2)-2026-03-06-UH.pdf
House: NA
Language: ENG
Date: 2026-03-06
Type: HANSARD
File location: hansard/01mfgh4vw5ff6kwu57l5glxmuwmneo6zsg.pdf
PDF URL: https://www.parliament.gov.za/storage/app/media/Docs/hansard/01mfgh4vw5ff6kwu57l5glxmuwmneo6zsg.pdf
Name: HAN-NCOP-2026-03-05-UH.pdf
House: NCOP
Language: ENG
Date: 2026-03-05
Type: HANSARD
File location: hansard/01mfgh4vw6p77fogtl4rbywcnoxrphclb3.pdf
PDF URL: https://www.parliament.gov.za/storage/app/media/Docs/hansard/01mfgh4vw6p77fogtl4rbywcnoxrphclb3.pdf
Name: HAN-NA-2026-03-05-UH.pdf
House: NA
Language: ENG
Date: 2026-03-05
Type: HANSARD
File location: hansard/01mfgh4vv7nz6y7l524rc3vinrur2dh42f.pdf
PDF URL: https://www.parliament.gov.za/storage/app/media/Docs/hansard/01mfgh4vv7nz6y7l524rc3vinrur2dh42f.pdf
Name: HAN-NA-2026-03-04-UH.pdf
House: NA
Language: ENG
Date: 2026-03-04
Type: HANSARD
File location: hansard/01mfgh4vufc3w4csruijhiftfar4hdyo5o.pdf
PDF URL: https://www.parliament.g

In [ ]:
 # ---- Read data back from Iceberg ----
rows = proceedings_table.scan().to_pandas()
rows

,id,uvimba_id,document_name,house,language,date,document_type,file_location,pdf_url,pdf_text


In [ ]:
print("Files currently in proceedings folder:")

response = s3_client.list_objects_v2(
    Bucket=bucket_name,
    Prefix="proceedings/"
)

for obj in response.get("Contents", []):
    print(obj["Key"])

Files currently in proceedings folder:
proceedings/2327197.pdf
proceedings/2327199.pdf
proceedings/HAN-JNT-2026-02-19-UH.pdf.pdf
proceedings/HAN-JNT-2026-02-19-UH.pdf
proceedings/HAN-MPS-NA(2)-2026-03-06-UH.pdf.pdf
proceedings/HAN-MPS-NA(2)-2026-03-06-UH.pdf
proceedings/HAN-NA-2026-02-24-UH.pdf.pdf
proceedings/HAN-NA-2026-02-24-UH.pdf
proceedings/HAN-NA-2026-02-25-UH.pdf.pdf
proceedings/HAN-NA-2026-02-25-UH.pdf
proceedings/HAN-NA-2026-03-03-UH.pdf.pdf
proceedings/HAN-NA-2026-03-03-UH.pdf
proceedings/HAN-NA-2026-03-04-UH.pdf.pdf
proceedings/HAN-NA-2026-03-04-UH.pdf
proceedings/HAN-NA-2026-03-05-UH.pdf.pdf
proceedings/HAN-NA-2026-03-05-UH.pdf
proceedings/HAN-NCOP-2026-02-27-UH.pdf.pdf
proceedings/HAN-NCOP-2026-02-27-UH.pdf
proceedings/HAN-NCOP-2026-03-04-UH.pdf.pdf
proceedings/HAN-NCOP-2026-03-04-UH.pdf
proceedings/HAN-NCOP-2026-03-05-UH.pdf.pdf
proceedings/HAN-NCOP-2026-03-05-UH.pdf


# Bills
# Intialize Warehouse connection

In [ ]:
warehouse_path = "/content/warehouse"
s3_uri = "https://15c46705015d147b9bbf2d532ce26495.r2.cloudflarestorage.com"
catalog = rest.RestCatalog(
    name="default",
    warehouse="15c46705015d147b9bbf2d532ce26495_standard-deviants-storage",
    uri="https://catalog.cloudflarestorage.com/15c46705015d147b9bbf2d532ce26495/standard-deviants-storage",
    token=userdata.get('CLOUDFLARE_API_KEY'),
)
if not catalog.namespace_exists("default"):
  catalog.create_namespace("default")

table_name = "default.bills"


# Create table using PyArrow schema
bill_schema = pa.schema([
    ("bill_identifier", pa.int64()),
    ("title", pa.string()),
    ("bill_number", pa.string()),
    ("pdf_url", pa.string()),
    ("pdf_text", pa.string()),
    ])
if not catalog.table_exists(table_name):
  bill_table = catalog.create_table(table_name, schema=bill_schema)
else:
  bill_table = catalog.load_table(table_name)

def retrieve_bill_by_id(idents: list[str]):
  rows = bill_table.scan(row_filter=In("bill_identifier", idents)).to_pandas()
  return rows

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=Exception('Connection closed while reading from the driver')>
Exception: Connection closed while reading from the driver


## Retrieve and load

In [ ]:
!pip install browserforge

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.5/726.5 kB 38.3 MB/s eta 0:00:00


In [ ]:
!playwright install chromium

In [ ]:
!apt-get install -y poppler-utils tesseract-ocr
!pip install pdfplumber pdf2image pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 124 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (2,433 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 122210 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from scrapling.fetchers import StealthyFetcher
import requests
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
import tempfile
import os
import re

def extract_pdf_text(url_or_path, is_url=True):
    """
    Extract PDF text using:
    - pdfplumber for initial extraction
    - OCR if text is empty or contains '(cid:'
    - fitz as fallback for normal pages
    """
    # Get PDF bytes
    if is_url:
        pdf_bytes = requests.get(url_or_path).content
    else:
        with open(url_or_path, "rb") as f:
            pdf_bytes = f.read()

    text_output = ""

    # First attempt: pdfplumber + OCR
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for i, page in enumerate(pdf.pages):


            page_text = page.extract_text()
            page_text_str = page_text if page_text else ""

            # Decide if OCR is needed
            if not page_text_str.strip() or '(cid:' in page_text_str:
                print(f"Page {i + 1} needs OCR...")

                images = convert_from_bytes(pdf_bytes, first_page=i+1, last_page=i+1, dpi=300)
                ocr_text = ""
                for img in images:
                    ocr_text += pytesseract.image_to_string(img) + "\n\n"

                page_text_str = ocr_text
            else:
                # fallback: use fitz for better normal text extraction
                doc = fitz.open(stream=pdf_bytes, filetype="pdf")
                fitz_page_text = doc[i].get_text()
                if fitz_page_text:
                    page_text_str = fitz_page_text

            # Clean spacing
            page_text_str = re.sub(r'\n+', '\n', page_text_str)                  # reduce multiple newlines
            page_text_str = re.sub(r' +', ' ', page_text_str)                    # reduce multiple spaces
            page_text_str = re.sub(r'([a-z])([A-Z])', r'\1 \2', page_text_str)  # fix missing spaces

            text_output += page_text_str.strip() + "\n\n"

    return text_output


start_bill_id = 1205
end_bill_id = 1335

StealthyFetcher.adaptive = True

documents = {}

for bill_id in range(start_bill_id, end_bill_id + 1):

    url = f"https://pmg.org.za/bill/{bill_id}/"

    p = await StealthyFetcher.async_fetch(
        url,
        headless=True,
        network_idle=True
    )

    article = p.css("article.page")[0]

    h1 = article.css("h1")[0]
    title_tag = p.css("title")[0].text.strip()

    title = title_tag.replace(" | PMG", "")

    bill_number = h1.css("span.text-muted")[0].text.strip()


    full_text = title.strip()

    title = full_text.replace(bill_number, "").strip()

    print("Bill ID:", bill_id)
    print("Title:", title)
    print("Bill Number:", bill_number)

    # find PDF
    pdf_links = article.css("a[href$='.pdf']")

    for link in pdf_links:
        pdf_url = link.attrib.get("href")

        print("PDF URL:", pdf_url)

        text = extract_pdf_text(pdf_url)

        print("\n----- PDF TEXT PREVIEW -----\n")
        print(text[:1000])
        print("\n----------------------------\n")

        filename = f"bill_{bill_id}_text.txt"
        with open(filename, "w", encoding="utf-8") as f:
            f.write(text)

        documents[bill_id] = {
            "bill_identifier": bill_id,
            "title": title,
            "bill_number": bill_number,
            "pdf_url": pdf_url,
            "pdf_text": text
        }

        break

if documents:
    df = pd.DataFrame(documents.values())
    # Convert to PyArrow and append to Iceberg
    arrow_table = pa.Table.from_pandas(df)
    bill_table.append(arrow_table)

    print(f"{len(df)} bills inserted into Iceberg table.")

    # ---- Read data back from Iceberg table ----
    rows = bill_table.scan().to_pandas()

    print("\nData currently in Iceberg table:\n")
    print(rows)

[2026-03-16 14:06:50] INFO: Fetched (200) <GET https://pmg.org.za/bill/1205/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://pmg.org.za/bill/1205/> (referer: https://www.google.com/)


Bill ID: 1205
Title: National State Enterprises Bill
Bill Number: (X-2024)
PDF URL: https://pmg.org.za/files/State_Enterprise_Bill.pdf

----- PDF TEXT PREVIEW -----

This gazette is also available free online at www.gpwonline.co.za
STAATSKOERANT, 9 Januarie 2024
No. 49978    11
DEPARTMENT OF PUBLIC ENTERPRISES
NO. 4242
9 January 2024

This gazette is also available free online at www.gpwonline.co.za
12    No. 49978	
GOVERNMENT GAZETTE, 9 January 2024

This gazette is also available free online at www.gpwonline.co.za
	
STAATSKOERANT, 9 Januarie 2024
No. 49978    13

This gazette is also available free online at www.gpwonline.co.za
14    No. 49978	
GOVERNMENT GAZETTE, 9 January 2024

This gazette is also available free online at www.gpwonline.co.za
	
STAATSKOERANT, 9 Januarie 2024
No. 49978    15

This gazette is also available free online at www.gpwonline.co.za
16    No. 49978	
GOVERNMENT GAZETTE, 9 January 2024

This gazette is also available free online at www.gpwonline.co.za
	
ST

[2026-03-16 14:06:58] INFO: Fetched (200) <GET https://pmg.org.za/bill/1206/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://pmg.org.za/bill/1206/> (referer: https://www.google.com/)


Bill ID: 1206
Title: Repeal of the South African Airways Bill
Bill Number: (X-2024)
PDF URL: https://pmg.org.za/files/SAA.pdf

----- PDF TEXT PREVIEW -----

This gazette is also available free online at www.gpwonline.co.za
	
STAATSKOERANT, 9 Januarie 2024
No. 49978    3
Government Notices • Goewermentskennisgewings
DEPARTMENT OF PUBLIC ENTERPRISES
NO. 4241
9 January 2024

This gazette is also available free online at www.gpwonline.co.za
4    No. 49978	
GOVERNMENT GAZETTE, 9 January 2024

This gazette is also available free online at www.gpwonline.co.za
	
STAATSKOERANT, 9 Januarie 2024
No. 49978    5

This gazette is also available free online at www.gpwonline.co.za
6    No. 49978	
GOVERNMENT GAZETTE, 9 January 2024

This gazette is also available free online at www.gpwonline.co.za
	
STAATSKOERANT, 9 Januarie 2024
No. 49978    7

This gazette is also available free online at www.gpwonline.co.za
8    No. 49978	
GOVERNMENT GAZETTE, 9 January 2024

This gazette is also available free o

[2026-03-16 14:07:06] INFO: Fetched (200) <GET https://pmg.org.za/bill/1207/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://pmg.org.za/bill/1207/> (referer: https://www.google.com/)


...

Bill ID: 1254
Title: Local Government: Municipal Structures Amendment Bill
Bill Number: (X-2025)
PDF URL: https://pmg.org.za/files/Local_Government__Municipal_Structures_Amendment_Bill__Draft__Comments_invited.pdf

----- PDF TEXT PREVIEW -----

This gazette is also available free online at www.gpwonline.co.za
STAATSKOERANT, 24 Januarie 2025
No. 51950    75
REPUBLIC OF SOUTH AFRICA 
LOCAL GOVERNMENT: MUNICIPAL 
STRUCTURES AMENDMENT BILL 
(As introduced in the National Assembly (proposed section 75); Bill and prior notice of its 
introduction published in Government Gazette No. XXXXXXX); 
(The English text is the official text of the Bill) 
(Mr G Michalakis, MP) 
[B x —2025] 
ISBN 
No. of copies printed ....................................................................

This gazette is also available free online at www.gpwonline.co.za
76    No. 51950	
GOVERNMENT GAZETTE, 24 January 2025
GENERAL EXPLANATORY NOTE: 
[ 
 ] 
 Words in bold type in square brackets indicate omissions from 
e

[2026-03-16 14:29:17] INFO: Fetched (200) <GET https://pmg.org.za/bill/1255/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://pmg.org.za/bill/1255/> (referer: https://www.google.com/)


Bill ID: 1255
Title: Economic Regulation of Transport Amendment Bill 2024
Bill Number: (X-2024)
PDF URL: https://pmg.org.za/files/241206draft-economic-regulation-of-transport-amendment-bill.pdf

----- PDF TEXT PREVIEW -----

This gazette is also available free online at www.gpwonline.co.za
230    No. 51711	
GOVERNMENT GAZETTE, 6 December 2024
REPUBLIC OF SOUTH AFRICA 
 
----------------- 
 
 
 
ECONOMIC REGULATION OF TRANSPORT AMENDMENT BILL 
 
 
 
----------------- 
 
(As introduced in the National Assembly (proposed section 75); Bill and prior notice of its 
introduction published in Government Gazette No. of 
 
) 
(The English text is the official text of the Bill) 
 
__________ 
 
 
(Portfolio Committee on Transport)

This gazette is also available free online at www.gpwonline.co.za
	
STAATSKOERANT, 6 Desember 2024
No. 51711    231
BILL 
 
To amend the Economic Regulation of Transport Act, 2024, so as to correct erroneous 
references in Schedule 1 to the Act; and to provide for ma

[2026-03-16 14:29:22] INFO: Fetched (404) <GET https://pmg.org.za/bill/1256/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (404) <GET https://pmg.org.za/bill/1256/> (referer: https://www.google.com/)


IndexError: list index out of range

## Show data loaded

In [ ]:
rows = bill_table.scan().to_pandas()



# Get the largest bill_identifier
max_bill_id = rows['bill_identifier'].max()

print("Largest bill_identifier:", max_bill_id)

min_bill_id = rows['bill_identifier'].min()
print("Smallest bill_identifier:", min_bill_id)

min_row = rows.loc[rows['bill_identifier'] == rows['bill_identifier'].min()]
print(min_row)

## Store PDFs

In [ ]:
import requests
import re
from io import BytesIO

def sanitize_proceedings_filename(name: str) -> str:
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = name.replace(" ", "_")
    return name

from io import BytesIO
import requests

def upload_bills_to_s3(bills, s3_client, bucket_name):

    uploaded = 0

    for bill in bills:
        try:

            pdf_url = bill["pdf_url"]
            bill_id = bill["bill_identifier"]

            filename = f"{bill_id}.pdf"
            s3_key = f"bills/{filename}"

            print(f"Uploading {filename}...")

            response = requests.get(pdf_url)

            if response.status_code != 200:
                print(f"Failed to download {pdf_url}")
                continue

            file_bytes = BytesIO(response.content)

            s3_client.upload_fileobj(
                file_bytes,
                bucket_name,
                s3_key
            )

            uploaded += 1

        except Exception as e:
            print(f"Upload failed for {bill['title']}: {e}")

    print(f"{uploaded} files uploaded to s3://{bucket_name}/bills/")

## Retreive bills from PMG

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pyiceberg.catalog import load_catalog
from pyiceberg.expressions import In
import httpx
import asyncio
import json


from scrapling.fetchers import StealthyFetcher
import requests
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
import tempfile
import os
import re

def extract_pdf_text(url_or_path, is_url=True):
    """
    Extract PDF text using:
    - pdfplumber for initial extraction
    - OCR if text is empty or contains '(cid:'
    - fitz as fallback for normal pages
    """
    # Get PDF bytes
    if is_url:
        pdf_bytes = requests.get(url_or_path).content
    else:
        with open(url_or_path, "rb") as f:
            pdf_bytes = f.read()

    text_output = ""

    # First attempt: pdfplumber + OCR
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for i, page in enumerate(pdf.pages):


            page_text = page.extract_text()
            page_text_str = page_text if page_text else ""

            # Decide if OCR is needed
            if not page_text_str.strip() or '(cid:' in page_text_str:
                print(f"Page {i + 1} needs OCR...")

                images = convert_from_bytes(pdf_bytes, first_page=i+1, last_page=i+1, dpi=300)
                ocr_text = ""
                for img in images:
                    ocr_text += pytesseract.image_to_string(img) + "\n\n"

                page_text_str = ocr_text
            else:
                # fallback: use fitz for better normal text extraction
                doc = fitz.open(stream=pdf_bytes, filetype="pdf")
                fitz_page_text = doc[i].get_text()
                if fitz_page_text:
                    page_text_str = fitz_page_text

            # Clean spacing
            page_text_str = re.sub(r'\n+', '\n', page_text_str)                  # reduce multiple newlines
            page_text_str = re.sub(r' +', ' ', page_text_str)                    # reduce multiple spaces
            page_text_str = re.sub(r'([a-z])([A-Z])', r'\1 \2', page_text_str)  # fix missing spaces

            text_output += page_text_str.strip() + "\n\n"

    return text_output


warehouse_path = "/content/warehouse"
s3_uri = "https://15c46705015d147b9bbf2d532ce26495.r2.cloudflarestorage.com"
catalog = rest.RestCatalog(
    name="default",
    warehouse="15c46705015d147b9bbf2d532ce26495_standard-deviants-storage",
    uri="https://catalog.cloudflarestorage.com/15c46705015d147b9bbf2d532ce26495/standard-deviants-storage",
    token=userdata.get('CLOUDFLARE_API_KEY'),
)
if not catalog.namespace_exists("default"):
  catalog.create_namespace("default")

table_name = "default.bills"


bill_schema = pa.schema([
    ("bill_identifier", pa.int64()),
    ("title", pa.string()),
    ("bill_number", pa.int64()),
    ("year", pa.int32()),
    ("act_name", pa.string()),
    ("introduced_by", pa.string()),
    ("date_of_introduction", pa.string()),
    ("date_of_assent", pa.string()),
    ("effective_date", pa.string()),
    ("status_name", pa.string()),
    ("type_name", pa.string()),
    ("events_json", pa.string()),
    ("pdf_url", pa.string()),
    ("pdf_text", pa.string()),
])


if not catalog.table_exists(table_name):
    bill_table = catalog.create_table(table_name, schema=bill_schema)
else:
    bill_table = catalog.load_table(table_name)



start_bill_id = 1208
end_bill_id = 1335

bills = []

for bill_id in range(start_bill_id, end_bill_id + 1):

    api_url = f"https://api.pmg.org.za/bill/{bill_id}/"
    print(f"Fetching {api_url}")

    try:
        async with httpx.AsyncClient() as client:
            response = await client.get(api_url)
            response.raise_for_status()
            bill_data = response.json()


        title = bill_data.get("title")
        bill_number = bill_data.get("number")
        year = bill_data.get("year")
        act_name = bill_data.get("act_name")
        introduced_by = bill_data.get("introduced_by")
        date_of_introduction = bill_data.get("date_of_introduction")
        date_of_assent = bill_data.get("date_of_assent")
        effective_date = bill_data.get("effective_date")
        status_name = bill_data.get("status", {}).get("name")
        type_name = bill_data.get("type", {}).get("name")

        pdf_url = None
        versions = bill_data.get("versions", [])
        if versions:
            pdf_url = versions[0].get("file", {}).get("url")

        filename = f"bill_{bill_id}_text.txt"
        # if os.path.exists(filename):
        #   print(f"Reading saved text for Bill {bill_id} from {filename}")

        #   with open(filename, "r", encoding="utf-8") as f:
        #       pdf_text = f.read()
        # else:
        pdf_text = ""
        if pdf_url:
            try:
                pdf_text = extract_pdf_text(pdf_url)

                # Save it for future runs
                with open(filename, "w", encoding="utf-8") as f:
                    f.write(pdf_text)

            except Exception as e:
                print(f"Error extracting PDF text: {e}")

            events_json = json.dumps(bill_data.get("events", []))
            full_response_json = json.dumps(bill_data)

        bills.append({
            "bill_identifier": bill_id,
            "title": title,
            "bill_number": bill_number,
            "year": year,
            "act_name": act_name,
            "introduced_by": introduced_by,
            "date_of_introduction": date_of_introduction,
            "date_of_assent": date_of_assent,
            "effective_date": effective_date,
            "status_name": status_name,
            "type_name": type_name,
            "events_json": events_json,
            "pdf_url": pdf_url,
            "pdf_text": pdf_text,
        })

    except httpx.HTTPStatusError as e:
        print(f"HTTP error for Bill ID {bill_id}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred for Bill ID {bill_id}: {e}")

if bills:
    df = pd.DataFrame(bills)
    arrow_table = pa.Table.from_pandas(df, schema=bill_schema, preserve_index=False)

    bill_table.append(arrow_table)
    print(f"{len(bills)} new bills inserted into Iceberg table '{table_name}'.")

    upload_bills_to_s3(bills, s3_client, bucket_name)


else:
    print("No new bills to insert.")



Fetching https://api.pmg.org.za/bill/1208/
Page 31 needs OCR...
Fetching https://api.pmg.org.za/bill/1209/
Fetching https://api.pmg.org.za/bill/1210/
Page 2 needs OCR...
Page 3 needs OCR...
Fetching https://api.pmg.org.za/bill/1211/
Page 96 needs OCR...
Page 106 needs OCR...
Page 110 needs OCR...
Fetching https://api.pmg.org.za/bill/1212/
Page 5 needs OCR...
Page 7 needs OCR...
Page 12 needs OCR...
Page 13 needs OCR...
Page 14 needs OCR...
Page 15 needs OCR...
Page 16 needs OCR...
Page 17 needs OCR...
Page 18 needs OCR...
Page 19 needs OCR...
Page 20 needs OCR...
Page 21 needs OCR...
Page 22 needs OCR...
Page 23 needs OCR...
Page 24 needs OCR...
Page 25 needs OCR...
Page 26 needs OCR...
Page 27 needs OCR...
Page 28 needs OCR...
Page 29 needs OCR...
Page 30 needs OCR...
Page 31 needs OCR...
Page 32 needs OCR...
Page 33 needs OCR...
Page 34 needs OCR...
Page 35 needs OCR...
Page 36 needs OCR...
Page 37 needs OCR...
Page 38 needs OCR...
Page 39 needs OCR...
Page 40 needs OCR...
Page 41 ne

## Show loaded data

In [ ]:
rows = bill_table.scan().to_pandas()
rows

,bill_identifier,title,bill_number,year,act_name,introduced_by,date_of_introduction,date_of_assent,effective_date,status_name,type_name,events_json,pdf_url,pdf_text
0,1208,National State Enterprises Bill,1.0,2024,None,Minister of Public Enterprises,2024-01-24,None,None,na,S75,"[{""created_at"": ""2025-07-09T09:28:22.961073+00...",https://pmg.org.za/files/B1-2024_National_Stat...,REPUBLIC OF SOUTH AFRICA\nNATIONAL STATE\nENTE...
1,1209,Repeal of South African Airways Bill,2.0,2024,None,Minister of Public Enterprises,2024-01-24,None,None,withdrawn,S75,"[{""created_at"": ""2024-01-25T05:37:26.481342+00...",https://pmg.org.za/files/B2-2024_Repeal_of_Sou...,REPUBLIC OF SOUTH AFRICA\nREPEAL OF\nSOUTH AFR...
2,1210,Pension Funds Amendment Bill,3.0,2024,Act 31 of 2024,Minister of Finance,2024-01-30,2024-07-18,2024-08-23,act-commenced,S75,"[{""created_at"": ""2024-01-31T11:28:51.693116+00...",https://pmg.org.za/files/50968_23-7_PensionsFu...,cure\nthe \nis \nPrevention \n0800-0123-22 \nH...
3,1211,Division of Revenue Bill,4.0,2024,Act 24 of 2024,Minister of Finance,2024-02-21,2024-06-03,2024-06-03,act-commenced,S76,"[{""created_at"": ""2024-02-21T09:25:57.278369+00...",https://pmg.org.za/files/50743_03-06_Divisiono...,cure\nthe \nis \nPrevention \n0800-0123-22 \nH...
4,1212,Appropriation Bill,5.0,2024,Act 40 of 2024,Minister of Finance,2024-02-21,2024-08-16,2024-08-16,act-commenced,S77,"[{""created_at"": ""2024-02-21T09:27:30.224460+00...",https://pmg.org.za/files/51101_ApproprriationA...,cure\nthe \nis \nPrevention \n0800-0123-22 \nH...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,1331,Traditional and Khoi-San Leadership Bill,NaN,2026,None,Minister of Cooperative Governance and Traditi...,2026-02-27,None,None,None,Draft,[],None,
113,1332,Gas Bill,6.0,2026,None,Minister of Electricity and Energy,2026-03-05,None,None,na,S76,"[{""created_at"": ""2026-03-06T10:03:22.548451+00...",https://pmg.org.za/files/B6-2026_Gas.pdf,REPUBLIC OF SOUTH AFRICA\nGAS BILL\n(As introd...
114,1333,Gas Bill,NaN,2026,None,Minister of Electricity and Energy,None,None,None,None,Draft,[],None,
115,1334,South African Geographical Names Council Amend...,NaN,2026,None,"Minister of Sport, Arts and Culture",2026-03-06,None,None,None,Draft,[],https://pmg.org.za/files/260306South_African_G...,\n \n \nREPUBLIC OF SOUTH AFRICA \n \n \n \n ...


## List stored PDFs

In [ ]:
print("Files currently in bills folder:")

response = s3_client.list_objects_v2(
    Bucket=bucket_name,
    Prefix="bills/"
)

for obj in response.get("Contents", []):
    print(obj["Key"])

Files currently in bills folder:
bills/1208.pdf
bills/1209.pdf
bills/1210.pdf
bills/1211.pdf
bills/1212.pdf
bills/1213.pdf
bills/1214.pdf
bills/1215.pdf
bills/1216.pdf
bills/1217.pdf
bills/1218.pdf
bills/1219.pdf
bills/1220.pdf
bills/1221.pdf
bills/1222.pdf
bills/1223.pdf
bills/1224.pdf
bills/1225.pdf
bills/1226.pdf
bills/1227.pdf
bills/1228.pdf
bills/1229.pdf
bills/1230.pdf
bills/1231.pdf
bills/1232.pdf
bills/1233.pdf
bills/1234.pdf
bills/1235.pdf
bills/1236.pdf
bills/1237.pdf
bills/1238.pdf
bills/1239.pdf
bills/1240.pdf
bills/1241.pdf
bills/1242.pdf
bills/1243.pdf
bills/1244.pdf
bills/1245.pdf
bills/1246.pdf
bills/1247.pdf
bills/1248.pdf
bills/1249.pdf
bills/1250.pdf
bills/1251.pdf
bills/1252.pdf
bills/1253.pdf
bills/1254.pdf
bills/1255.pdf
bills/1261.pdf
bills/1262.pdf
bills/1263.pdf
bills/1264.pdf
bills/1265.pdf
bills/1266.pdf
bills/1267.pdf
bills/1268.pdf
bills/1269.pdf
bills/1270.pdf
bills/1271.pdf
bills/1272.pdf
bills/1273.pdf
bills/1274.pdf
bills/1275.pdf
bills/1276.pdf
bills/1